# Train wav2vec2 Quran ASR + corrector - Colab

Use **Runtime > Change runtime type > GPU** before running. Audio, processed dataset, and HF caches stay on `/content`; trained checkpoints are saved to Google Drive.

## 0. Put HF caches on `/content`

In [1]:
from pathlib import Path
import os

cwd = Path.cwd().resolve()

print("Current working directory:")
print(cwd)

print("\nIsi folder sekarang:")
for item in sorted(cwd.iterdir()):
    print("-", item.name)

Current working directory:
/home/jrilym/Projects/AI/model-asr-quran

Isi folder sekarang:
- .cache
- .git
- .gitignore
- .pytest_cache
- .python-version
- .ruff_cache
- .tokensave
- .venv
- 00_colab_test.ipynb
- 01_colab_train.ipynb
- Makefile
- README.id.md
- README.md
- backups
- checkpoints
- configs
- data
- notebooks
- outputs
- pyproject.toml
- quran_asr
- requirements-colab.txt
- scripts
- tests
- uv.lock


In [2]:
from pathlib import Path
import os


if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "quran_asr").exists() and (cwd / "configs").exists():
    PROJECT_ROOT = cwd
else:
    raise RuntimeError(f"Posisi folder salah: {cwd}")

os.chdir(PROJECT_ROOT)

print("Project root:", Path.cwd())

Project root: /home/jrilym/Projects/AI/model-asr-quran


## 1. Get the code

In [3]:
from pathlib import Path
import os

cwd = Path.cwd().resolve()

if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
elif (cwd / "quran_asr").exists() and (cwd / "configs").exists():
    PROJECT_ROOT = cwd
else:
    raise RuntimeError(f"Posisi folder salah: {cwd}")

os.chdir(PROJECT_ROOT)

print("Project root:", Path.cwd())
print("Ada configs:", (PROJECT_ROOT / "configs").exists())
print("Ada quran_asr:", (PROJECT_ROOT / "quran_asr").exists())
print("Ada data:", (PROJECT_ROOT / "data").exists())

Project root: /home/jrilym/Projects/AI/model-asr-quran
Ada configs: True
Ada quran_asr: True
Ada data: True


## 2. Install dependencies and verify GPU

In [4]:
!bash scripts/install_colab_deps.sh
!python -m pip install -q -e . --no-deps
!python scripts/cloud_preflight.py


Using existing CUDA PyTorch:
  torch=2.12.1+cu130 cuda=13.0 device=NVIDIA GeForce RTX 3050 6GB Laptop GPU
  torchaudio=2.11.0+cu130
scripts/install_colab_deps.sh: line 26: apt-get: command not found

=== cloud pre-flight ===
  [OK  ] CUDA GPU — NVIDIA GeForce RTX 3050 6GB Laptop GPU
  [OK  ] ffmpeg — /usr/bin/ffmpeg
  [OK  ] dep accelerate
  [OK  ] dep torch
  [OK  ] dep torchaudio
  [OK  ] dep transformers
  [OK  ] dep datasets
  [OK  ] dep evaluate
  [OK  ] dep jiwer
  [OK  ] dep soundfile
  [OK  ] dep librosa
  [OK  ] dep huggingface_hub
  [OK  ] dep yaml
  [OK  ] dep requests
  [OK  ] dep tqdm
  [WARN] HF token — none — set logging.hub_repo=null or run notebook_login()
  [OK  ] disk free — 237.0 GB free on / (need ~15-20 GB)

0 hard failure(s). Fix FAIL items before training.


In [5]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
DATA_DIR = PROJECT_ROOT / "data"
METRICS_DIR = PROJECT_ROOT / "outputs"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

print("Checkpoint dir:", CHECKPOINT_DIR)
print("Data dir:", DATA_DIR)
print("Metrics dir:", METRICS_DIR)

Checkpoint dir: /home/jrilym/Projects/AI/model-asr-quran/checkpoints
Data dir: /home/jrilym/Projects/AI/model-asr-quran/data
Metrics dir: /home/jrilym/Projects/AI/model-asr-quran/outputs


In [6]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", torch.cuda.get_device_properties(0).total_memory / 1024**3)

Torch: 2.12.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM GB: 5.6676025390625


## 4. Provision audio

In [9]:
import os
import subprocess
from pathlib import Path

def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)

PROJECT_ROOT = Path.cwd()

audio_dir = PROJECT_ROOT / "data" / "raw" / "audio"
backup_dir = PROJECT_ROOT / "backups"
backup = backup_dir / "audio.tar"

audio_dir.parent.mkdir(parents=True, exist_ok=True)
backup_dir.mkdir(parents=True, exist_ok=True)

has_audio = audio_dir.is_dir() and any(audio_dir.iterdir())
has_backup = backup.exists()

def backup_audio() -> None:
    print("Backing up audio locally...")
    sh(f'tar cf "{backup}" -C "{audio_dir.parent}" audio')

if has_audio and has_backup:
    print("Audio and local backup are already present.")
elif has_audio:
    backup_audio()
elif has_backup:
    print("Restoring audio from local backup...")
    audio_dir.parent.mkdir(parents=True, exist_ok=True)
    sh(f'tar xf "{backup}" -C "{audio_dir.parent}"')
else:
    print("First run: downloading MP3 audio...")
    sh("python scripts/download.py --config configs/cloud_local.yaml --no-convert --max-workers 8 --qps 10")
    backup_audio()

First run: downloading MP3 audio...


[text] cache ready: data/raw/text/quran_uthmani.json
[audio] 6236 ayat × 2 reciter(s)
Husary_128kbps_Mujawwad: 100%|██████████| 6236/6236 [20:46<00:00,  5.00it/s]  
[Husary_128kbps_Mujawwad] {'downloaded': 6236}
Minshawy_Murattal_128kbps: 100%|██████████| 6236/6236 [17:03<00:00,  6.09it/s]  
[Minshawy_Murattal_128kbps] {'downloaded': 6236}


Backing up audio locally...


## 5. Build dataset and vocab

In [10]:
!python scripts/build.py --config configs/cloud_local.yaml
!python scripts/build_vocab.py --config configs/cloud_local.yaml


Saving the dataset (1/1 shards): 100%|█| 11348/11348 [00:00<00:00, 178985.74 ex
Saving the dataset (1/1 shards): 100%|█| 28/28 [00:00<00:00, 11035.57 examples/
Saving the dataset (1/1 shards): 100%|█| 108/108 [00:00<00:00, 36238.79 example
  train: 11348 rows
  validation: 28 rows
  test: 108 rows
[build] saved DatasetDict to data/processed
Traceback (most recent call last):
  File "/usr/lib/python3.12/pathlib.py", line 1311, in mkdir
    os.mkdir(self, mode)
FileNotFoundError: [Errno 2] No such file or directory: '/data/artifacts'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/jrilym/Projects/AI/model-asr-quran/scripts/build_vocab.py", line 53, in <module>
    raise SystemExit(main())
                     ^^^^^^
  File "/home/jrilym/Projects/AI/model-asr-quran/scripts/build_vocab.py", line 44, in main
    out = save_vocab(vocab, cfg.model.vocab_path)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/

## 6. Train

In [ ]:
from quran_asr.config import Config
from quran_asr.training.train import train

cfg = Config.from_yaml("configs/cloud_local.yaml")
trainer, processor = train(cfg, resume=True)
print("training done")


## 7. Evaluate

In [ ]:
!python scripts/evaluate.py --config configs/cloud_local.yaml --split test --out /content/quran_asr/metrics.json

## 8. Corrector demo

In [ ]:
from pathlib import Path

audio_file = next(Path("/content/quran_asr/data/raw/audio/Husary_128kbps_Mujawwad").glob("001001.*"))
!python scripts/correct.py --model-dir /content/drive/MyDrive/quran_asr/checkpoints --device cuda --audio "$audio_file" --text "بِسْمِ ٱللَّهِ ٱلرَّحْمَـٰنِ ٱلرَّحِيمِ"


## Resume after disconnect

Rerun cells 0 through 3, then cell 4 restores audio from Drive if the `/content` copy disappeared. Cell 6 uses `resume=True`, so training continues from the latest checkpoint in `/content/drive/MyDrive/quran_asr/checkpoints`.